# Alert 생성기 실험 ⑤ — 룰만으로 Alert 만들기

생성기를 거치지 않는다. 룰이 고른 거래를 **대상별로 모은 것을 그대로 Alert 로** 만들고, 그 Alert 가 데이터셋의 **세탁 시도**(Patterns 파일의 BEGIN ~ END 한 덩어리)와 얼마나 같은지 본다.
지표는 02번과 같고, 모델 씨앗으로 생성기가 만든 Alert(01번)를 옆에 둔다.

| 방법 | 만드는 규칙 |
|---|---|
| G 생성기 | 01번 결과(`members.parquet`). 모델 씨앗 → 서비스 Alert 생성기 |
| R1 적중마다 | 룰 적중 묶음 하나가 Alert 하나. 같은 거래가 여러 Alert 에 들 수 있다 |
| R2 겹치면 합침 | 적중 묶음끼리 거래를 하나라도 공유하면 한 Alert 로 합친다. 한 거래는 한 Alert 에만 든다 |
| R3 적중마다 · 주변 거래 | R1 과 같되, Alert 에 근거 거래뿐 아니라 **대상 계좌의 주변 거래**까지 넣는다 (3-1절) |
| R4 겹치면 합침 · 주변 거래 | R2 와 같되, 주변 거래까지 넣은 뒤에 합친다 |

**적중 묶음** = 한 룰이 같은 대상 때문에 고른 거래들 (03번 9-7절, `rule_lib.hit_groups`).
- FAN-OUT · FAN-IN · GATHER-SCATTER: 대상 계좌(뿌리는 / 모으는 / 가운데 계좌)가 같은 근거 거래
- SCATTER-GATHER: 도착 계좌가 같은 근거 거래
- 위 넷은 대상이 같아도, 이웃한 두 근거 거래의 간격이 그 룰의 기간을 넘으면 나눈다
- CYCLE · RANDOM: 근거 거래끼리 "이어지는 쌍"으로 연결된 덩어리

- 03번(9-7절까지)과 01번을 같은 DATASET 으로 먼저 돌려야 한다. 룰도 생성기도 여기서 다시 돌리지 않는다.
- 정답, 제외 패턴, "묶음은 거래 2건 이상" 규칙은 02번과 같다.
- 룰 적중 묶음은 읽은 기간 전체에서 만든 것이고, 여기서는 그중 **평가 구간 안의 거래**만 쓴다. 생성기도 평가 구간의 씨앗만 받았다.

## 0. 설정

In [ ]:
DATASET = 'HI-Medium'            # 03번과 01번을 같은 DATASET 으로 먼저 돌려야 한다
RULE_RUN = 'rule_vs_model'               # 03번 결과 폴더 (hit_groups.parquet, hits.parquet)
GEN_RUN = 'before_after_grouping'        # 01번 결과 폴더 (members.parquet)
EXCLUDE = ['STACK', 'BIPARTITE']         # 평가에서 빼는 패턴 (02번과 같다)

In [ ]:
from pathlib import Path
import json
import duckdb, numpy as np, pandas as pd
import gen_lib as GL
import rule_lib as RL

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir() or (p / 'datasets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
RULE_DIR = Path.cwd() / 'results' / DATASET / RULE_RUN
GEN_DIR = Path.cwd() / 'results' / DATASET / GEN_RUN
OUT = Path.cwd() / 'results' / DATASET / 'rule_alerts'
OUT.mkdir(parents=True, exist_ok=True)
rule_run = json.load(open(RULE_DIR / 'run.json', encoding='utf-8'))
gen_run = json.load(open(GEN_DIR / 'run.json', encoding='utf-8'))
assert rule_run['test_window'] == gen_run['test_window'], '03번과 01번의 평가 구간이 다르다'
a, b = rule_run['test_window']
print('평가 구간', a, '~', b, '| 01번 점수', gen_run['scores'], '| 기준', gen_run['threshold'])
pd.DataFrame(rule_run['chosen']).T.rename(index=RL.RULES)      # 03번이 고른 룰 조건

## 1. 룰 적중 묶음 읽기
`hit_groups.parquet` 의 한 줄은 (거래, 룰, 대상, 묶음)이다. 시각과 계좌는 거래 파일에서, 세탁 여부와 패턴은 03번의 `hits.parquet` 에서 붙인다.

In [ ]:
groups = pd.read_parquet(RULE_DIR / 'hit_groups.parquet')
con = duckdb.connect()
con.register('group_ids', groups[['edge_id']].drop_duplicates())
tx = con.execute(f"""SELECT t.edge_id, t.ts, t.src, t.dst
                     FROM read_parquet('{BASE / 'transactions.parquet'}') t JOIN group_ids USING (edge_id)""").df()
groups = groups.merge(tx, on='edge_id', how='left')
assert groups.ts.notna().all(), '거래 파일에 없는 edge_id 가 있다'
start, end = pd.Timestamp(a), pd.Timestamp(b) + pd.Timedelta(days=1)
print(f'읽은 기간 전체: 거래 {groups.edge_id.nunique():,} 건, 적중 묶음 {groups[["룰", "묶음"]].drop_duplicates().shape[0]:,} 개')
groups = groups[(groups.ts >= start) & (groups.ts < end)]                    # 평가 구간 안의 거래만 쓴다

In [ ]:
hits = pd.read_parquet(RULE_DIR / 'hits.parquet')
label = hits.set_index('edge_id')[['y', 'pattern_type']]
groups = groups.join(label, on='edge_id')
assert groups.y.notna().all(), 'hits.parquet 에 없는 거래가 있다. 03번을 9-7절까지 다시 돌려야 한다'
groups['y'] = groups.y.astype(int)
ORDER = [name for name in RL.RULES.values() if name in set(groups.룰)]
size = groups.groupby(['룰', '묶음']).size()
(size.groupby('룰').agg(적중_묶음='size', 거래='sum', 크기_중앙값='median', 크기_최대='max', 거래_1건짜리=lambda s: int((s == 1).sum()))
 .join(groups.groupby('룰').y.mean().round(4).rename('세탁_비율')).loc[ORDER])

## 2. 정답: 세탁 시도
02번과 같은 방법이다. 시도마다 평가 구간 안의 거래 수를 세고, 구간 안 거래가 2건 이상인 시도를 평가한다.

In [ ]:
PAT = BASE / 'patterns.parquet'
IN = f"ts >= TIMESTAMP '{start}' AND ts < TIMESTAMP '{end}'"
attempts = con.execute(f"""SELECT attempt_id, any_value(pattern_type) AS pattern_type, count(*) AS n_rows,
                                  count(*) FILTER (WHERE {IN}) AS n_in, count(edge_id) FILTER (WHERE {IN}) AS n_in_matched,
                                  min(ts) AS first_ts, max(ts) AS last_ts
                           FROM read_parquet('{PAT}') GROUP BY attempt_id""").df()
attempts['eligible'] = (~attempts.pattern_type.isin(EXCLUDE) & (attempts.n_in >= 2) & (attempts.n_in == attempts.n_in_matched))
# 거래 → 시도. 한 거래가 두 시도에 들어 있으면 번호가 작은 시도로 둔다
attempt_of = con.execute(f"""SELECT edge_id, min(attempt_id) AS attempt_id, arg_min(pattern_type, attempt_id) AS pattern_type,
                                    bool_and({IN}) AS in_window
                             FROM read_parquet('{PAT}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
pattern_all = attempt_of.set_index('edge_id').pattern_type          # 거래 → 패턴 (제외 패턴 포함). 3-1절에서 쓴다
attempt_of = attempt_of[~attempt_of.pattern_type.isin(EXCLUDE)]
eval_attempts = set(attempts.loc[attempts.eligible, 'attempt_id'])
truth = attempt_of[attempt_of.attempt_id.isin(eval_attempts) & attempt_of.in_window]      # 정답: 평가할 시도의 구간 안 거래
print(f'평가할 시도 {len(eval_attempts):,} 개 | 정답 거래 {len(truth):,} 건')

In [ ]:
# 묶기 전의 상한: 시도의 거래를 애초에 몇 건 골랐나. 고르지 못한 거래는 어떤 방법으로도 Alert 에 못 들어간다(생성기의 연결 거래 제외)
members = pd.read_parquet(GEN_DIR / 'members.parquet')
is_seed = members.seed_model | members.promoted          # 모델 씨앗 + 보강으로 들어온 씨앗 (02번과 같다)
picked = {'모델 씨앗': set(members.loc[is_seed, 'edge_id']), '룰이 고른 거래': set(groups.edge_id)}
rows = []
for name, ids in picked.items():
    t = truth.assign(hit=truth.edge_id.isin(ids)).groupby('attempt_id').agg(size=('hit', 'size'), hit=('hit', 'sum'))
    rows.append(dict(고른_것=name, 시도_수=len(t), 정답_거래=int(t['size'].sum()), 고른_거래=int(t.hit.sum()),
                     거래_재현율=round(float(t.hit.sum() / t['size'].sum()), 4), 전부_고른_시도=int((t.hit == t['size']).sum()),
                     두_건_이상_고른_시도=int((t.hit >= 2).sum()), 하나도_못_고른_시도=int((t.hit == 0).sum())))
ceiling = pd.DataFrame(rows).set_index('고른_것')
ceiling

## 3. Alert 만들기
- **R1 적중마다**: (룰, 묶음) 하나가 Alert 하나다. GATHER-SCATTER 의 모으는 쪽을 FAN-IN 룰도 고르는 식으로, 같은 거래가 Alert 여러 개에 들 수 있다.
- **R2 겹치면 합침**: 거래를 하나라도 공유하는 적중 묶음을 한 Alert 로 합친다. 공유가 사슬처럼 이어지면 전부 한 Alert 가 된다.
- Alert 는 **거래 2건 이상**만 센다(02번과 같다). 평가 구간 안에 근거 거래가 1건뿐인 적중 묶음은 빠진다.
- 제외 패턴의 거래는 합친 뒤에 뺀다(02번이 생성기 결과를 다루는 방식과 같다).

In [ ]:
groups['r1'] = pd.factorize(pd.MultiIndex.from_frame(groups[['룰', '묶음']]))[0]      # 적중 묶음 하나 = Alert 하나
groups['r2'] = RL.merge_groups(groups)                                              # 거래를 공유하는 적중 묶음끼리 합친 번호
kept = groups[~groups.pattern_type.isin(EXCLUDE)]

def as_groups(frame, column):
    g = frame[['edge_id', column, 'y']].drop_duplicates(['edge_id', column]).rename(columns={column: 'group'})
    return g[g.groupby('group')['edge_id'].transform('size') >= 2]

methods = {'G 생성기': as_groups(members[(members.role > 0) & ~members.pattern_type.isin(EXCLUDE)], 'alert'),
           'R1 적중마다': as_groups(kept, 'r1'), 'R2 겹치면 합침': as_groups(kept, 'r2')}

### 3-1. 주변 거래로 넓히기 (R3 · R4)
R1 · R2 의 Alert 에는 **필터(금액 · 첫거래 · 환전)를 통과한 근거 거래만** 들어 있다. 필터는 대상을 찾기 위한 조건이지 Alert 에 넣을 거래를 고르는 조건이 아니다.
예를 들어 FAN-IN 룰은 환전 직후 송금만 세므로, 모으는 계좌를 찾고도 그 계좌로 들어온 환전 아닌 세탁 거래는 Alert 에 넣지 못한다.

그래서 대상을 찾은 뒤에는 **그 대상의 거래를 필터와 무관하게** 넣어 본다 (`rule_lib.expand_groups`).
- 기간: 그 적중 묶음의 첫 근거 거래 시각 ~ 마지막 근거 거래 시각. 그 밖의 거래는 넣지 않는다
- FAN-OUT: 대상 계좌가 보낸 거래 / FAN-IN: 받은 거래 / GATHER-SCATTER: 받은 거래와 보낸 거래
- SCATTER-GATHER: 경유 계좌 → 도착 계좌 거래, 출발 계좌 → 경유 계좌 거래
- CYCLE · RANDOM: 넓히지 않는다
- 대가: 대상 계좌의 정상 거래도 같이 들어온다. 아래 표의 `추가_세탁_비율` 로 본다

In [ ]:
# 넓힐 때 볼 거래: 평가 구간 안에서, 적중 묶음에 나온 계좌가 보내거나 받은 거래 (자기 자신에게 보낸 거래 제외)
accounts = pd.DataFrame(dict(a=pd.unique(np.concatenate([groups.src.to_numpy(), groups.dst.to_numpy()]))))
con.register('accounts', accounts)
ledger = con.execute(f"""SELECT t.edge_id, t.ts, t.src, t.dst, t.is_laundering::INT AS y
                         FROM read_parquet('{BASE / 'transactions.parquet'}') t
                         WHERE {IN} AND t.src <> t.dst
                           AND (t.src IN (SELECT a FROM accounts) OR t.dst IN (SELECT a FROM accounts))""").df()
wide = RL.expand_groups(groups, ledger)
wide = wide.merge(ledger[['edge_id', 'y']], on='edge_id', how='left')
assert wide.y.notna().all()
wide['pattern_type'] = wide.edge_id.map(pattern_all).fillna('')
check = wide[wide.근거].merge(groups[['edge_id', '룰', '묶음', 'y']], on=['edge_id', '룰', '묶음'], suffixes=('', '_hits'))
assert len(check) == len(groups) and (check.y == check.y_hits).all(), '근거 거래가 빠졌거나 세탁 표시가 hits.parquet 와 다르다'
print(f'계좌 {len(accounts):,} 개의 거래 {len(ledger):,} 건을 봄 | 근거 거래 {len(groups):,} 줄 → 넓힌 뒤 {len(wide):,} 줄')

In [ ]:
# 넓혀서 들어온 거래가 얼마나 되고, 그중 세탁이 얼마나 되나
extra = (wide.groupby('룰').agg(근거_거래=('근거', 'sum'), 근거_세탁=('y', lambda s: int(s[wide.loc[s.index, '근거']].sum())),
                                추가_거래=('근거', lambda s: int((~s).sum())), 추가_세탁=('y', lambda s: int(s[~wide.loc[s.index, '근거']].sum())))
         .reindex(ORDER))
extra['근거_세탁_비율'] = (extra.근거_세탁 / extra.근거_거래).round(4)
extra['추가_세탁_비율'] = (extra.추가_세탁 / extra.추가_거래.replace(0, np.nan)).round(4)
extra

In [ ]:
wide['r1'] = pd.factorize(pd.MultiIndex.from_frame(wide[['룰', '묶음']]))[0]
wide['r2'] = RL.merge_groups(wide)                                                  # 넓힌 뒤에 합친다. 주변 거래를 공유해도 합쳐진다
kept_wide = wide[~wide.pattern_type.isin(EXCLUDE)]
methods['R3 적중마다 · 주변 거래'] = as_groups(kept_wide, 'r1')
methods['R4 겹치면 합침 · 주변 거래'] = as_groups(kept_wide, 'r2')
lonely = {'G 생성기': int((is_seed & ~members.pattern_type.isin(EXCLUDE)).sum()) - int(methods['G 생성기'].edge_id.isin(picked['모델 씨앗']).sum())}
lonely.update({name: (kept_wide if '주변' in name else kept).edge_id.nunique() - g.edge_id.nunique() for name, g in methods.items() if name != 'G 생성기'})
print('고른 거래 중 Alert 에 못 든 것(혼자 남음):', lonely)
pd.DataFrame({name: dict(Alert=g.group.nunique(), 거래=g.edge_id.nunique()) for name, g in methods.items()})

## 4. 정답과 비교

In [ ]:
summaries, per_attempt = [], {}
for name, g in methods.items():
    summary, per = RL.compare_alerts(name, g, attempt_of, eval_attempts, truth)
    summaries.append(summary)
    per_attempt[name] = per
result = pd.DataFrame(summaries).set_index('방법').T.astype(object)   # 정수는 정수로 보이게
result

In [ ]:
# 확인: G 생성기 줄은 02번 결과와 같아야 한다 (02번을 돌려 둔 경우)
check = Path.cwd() / 'results' / DATASET / 'grouping_vs_truth' / 'summary.csv'
if check.exists():
    old = pd.read_csv(check, index_col=0)['G 생성기']
    diff = {k: (old[k], result.loc[k, 'G 생성기']) for k in old.index if k in result.index and abs(float(old[k]) - float(result.loc[k, 'G 생성기'])) > 1e-9}
    print('02번과 같다' if not diff else f'02번과 다른 항목: {diff}')

**읽는 법** (02번과 같은 항목)
- `여러_묶음에_든_거래`: Alert 두 개 이상에 든 거래의 중복 수. 합치지 않는 R1 · R3 에서만 0 이 아니다.
- `세탁_비율`: Alert 에 든 거래 중 세탁 거래의 비율.
- `시도_2개이상_섞인_묶음`: 서로 다른 세탁 시도가 한 Alert 에 들어간 수.
- `쌍_정밀도`: 같은 Alert 에 든 두 거래가 실제로 같은 시도일 비율. 큰 Alert 에 이것저것 섞이면 떨어진다.
- `쌍_재현율`: 같은 시도의 두 거래가 같은 Alert 에 들어갔을 비율.
- `온전히_한_묶음`: 시도의 (구간 안) 거래 전부가 한 Alert 에 들어간 시도 수. 2절의 `전부_고른_시도` 가 상한이다.
- `평균_최대_포함률`: 시도마다 가장 많이 담은 Alert 하나가 그 시도 거래의 몇 %를 담았는지의 평균.
- `쪼개진_시도`: 시도의 거래가 Alert 2개 이상에 들어 있는 수. R1 · R3 에서는 같은 거래가 두 Alert 에 겹쳐 든 경우도 여기에 들어간다.
- `전혀_안_묶인_시도`: 시도의 거래가 어느 Alert 에도 안 든 수.

## 5. 패턴별

In [ ]:
def table_by(keys, per_map=None):
    frames = []
    for name, per in (per_map or per_attempt).items():
        f = per.groupby(keys, observed=True).agg(시도=('size', 'size'), 온전=('coverage', lambda s: round(float((s == 1).mean()), 3)),
                                                 포함률=('coverage', lambda s: round(float(s.mean()), 3)),
                                                 쪼개짐=('n_groups', lambda s: round(float((s >= 2).mean()), 3)),
                                                 안_묶임=('covered', lambda s: round(float((s == 0).mean()), 3)))
        frames.append(f.add_prefix(name.split()[0] + '_'))
    return pd.concat(frames, axis=1)

by_pattern = table_by('pattern_type')
by_pattern

## 5-1. 시도 크기별
크기 = 평가 구간 안 거래 수.

In [ ]:
SIZE_BINS, SIZE_LABELS = [1, 2, 5, 10, 20, 10_000], ['2건', '3~5건', '6~10건', '11~20건', '21건 이상']
by_size = table_by('크기', {name: per.assign(크기=pd.cut(per['size'], SIZE_BINS, labels=SIZE_LABELS)) for name, per in per_attempt.items()})
by_size

## 5-2. 평가 구간 뒤로 이어지는 시도 vs 구간 안에 끝난 시도
평가 구간이 끝난 뒤에도 거래가 이어지는 시도는 구간 안에서 **완성되기 전 모습**만 보인다. 룰은 완성된 모양을 찾기 때문에 여기서 더 불리하다.

In [ ]:
after = attempts.set_index('attempt_id').last_ts >= end          # 시도 → 구간 뒤로 이어지는가
with_tail = {name: per.assign(끝=per.attempt_id.map(after).map(GL.TAIL_LABELS)) for name, per in per_attempt.items()}
by_tail = table_by('끝', with_tail)
by_tail

In [ ]:
# 패턴 × 끝
by_pattern_tail = table_by(['pattern_type', '끝'], with_tail)
by_pattern_tail

## 6. 룰 Alert 들여다보기
6절과 7절은 근거 거래만으로 합친 R2 를 본다.
### 6-1. 합친 Alert(R2)는 어떤 룰들로 이루어졌나

In [ ]:
r2 = kept[kept.r2.isin(methods['R2 겹치면 합침'].group)]
alerts = r2.groupby('r2').agg(거래=('edge_id', 'nunique'), 적중_묶음=('r1', 'nunique'), 룰_수=('룰', 'nunique'),
                              룰=('룰', lambda s: ' + '.join(name for name in ORDER if name in set(s))))
alerts = alerts.join(r2.drop_duplicates(['r2', 'edge_id']).groupby('r2').y.sum().rename('세탁'))
alerts = alerts.join(r2.merge(attempt_of[['edge_id', 'attempt_id']], on='edge_id').groupby('r2').attempt_id.nunique().rename('시도_수')).fillna({'시도_수': 0})
alerts['시도_수'] = alerts.시도_수.astype(int)
alerts.groupby('룰_수').agg(Alert=('거래', 'size'), 거래_합=('거래', 'sum'), 크기_중앙값=('거래', 'median'), 크기_최대=('거래', 'max'),
                            세탁_비율=('세탁', lambda s: round(float(s.sum() / alerts.loc[s.index, '거래'].sum()), 4)),
                            시도_2개이상=('시도_수', lambda s: int((s >= 2).sum())))

In [ ]:
# 어떤 룰 조합이 많은가 (위 10개)
combo = alerts.groupby('룰').agg(Alert=('거래', 'size'), 거래_합=('거래', 'sum'), 크기_중앙값=('거래', 'median'),
                                 세탁_비율=('세탁', lambda s: round(float(s.sum() / alerts.loc[s.index, '거래'].sum()), 4)),
                                 시도_2개이상=('시도_수', lambda s: int((s >= 2).sum()))).sort_values('Alert', ascending=False)
combo.head(10)

### 6-2. 큰 Alert
방법마다 가장 큰 Alert 5개. 합치는 방식(R2)은 계좌를 타고 이어지면 커질 수 있다.

In [ ]:
rows = []
for name, g in methods.items():
    m = g.merge(attempt_of[['edge_id', 'attempt_id']], on='edge_id', how='left')
    stat = m.groupby('group').agg(거래=('edge_id', 'size'), 세탁=('y', 'sum'), 시도_수=('attempt_id', 'nunique'))
    for group, r in stat.sort_values('거래', ascending=False).head(5).iterrows():
        rows.append(dict(방법=name, 거래=int(r.거래), 세탁=int(r.세탁), 시도_수=int(r.시도_수),
                         룰=alerts.loc[group, '룰'] if name.startswith('R2') else ''))
largest = pd.DataFrame(rows)
largest

## 7. 생성기 Alert 와 겹침
룰 Alert(R2)가 모델 씨앗 · 생성기 Alert 와 얼마나 겹치는지 본다. 모델 씨앗이 하나도 없는 룰 Alert 가 **룰만으로 새로 생기는 Alert** 다.

In [ ]:
gen = methods['G 생성기']
alert_tx = methods['R2 겹치면 합침'].copy()
alert_tx['모델_씨앗'] = alert_tx.edge_id.isin(picked['모델 씨앗'])
alert_tx['생성기_Alert'] = alert_tx.edge_id.map(gen.drop_duplicates('edge_id').set_index('edge_id').group)
per_alert = alert_tx.groupby('group').agg(거래=('edge_id', 'size'), 세탁=('y', 'sum'), 모델_씨앗=('모델_씨앗', 'sum'),
                                          걸친_생성기_Alert=('생성기_Alert', 'nunique'))
per_alert['모델과'] = np.select([per_alert.모델_씨앗 == 0, per_alert.모델_씨앗 == per_alert.거래], ['씨앗 없음 (룰만)', '전부 씨앗'], '일부 씨앗')
overlap = per_alert.groupby('모델과').agg(Alert=('거래', 'size'), 거래=('거래', 'sum'), 세탁=('세탁', 'sum'),
                                          세탁_0건_Alert=('세탁', lambda s: int((s == 0).sum())),
                                          생성기_Alert_2개이상에_걸침=('걸친_생성기_Alert', lambda s: int((s >= 2).sum())))
overlap['세탁_비율'] = (overlap.세탁 / overlap.거래).round(4)
overlap

In [ ]:
# 거래 단위: 두 방법의 Alert 에 든 거래가 얼마나 같은가
rule_ids, gen_ids = set(alert_tx.edge_id), set(gen.edge_id)
y_of = pd.concat([alert_tx[['edge_id', 'y']], gen[['edge_id', 'y']]]).drop_duplicates('edge_id').set_index('edge_id').y
tx_overlap = pd.DataFrame([dict(거래=name, 건수=len(ids), 세탁=int(y_of.loc[list(ids)].sum()),
                                세탁_비율=round(float(y_of.loc[list(ids)].mean()), 4) if ids else None)
                           for name, ids in [('둘 다의 Alert 에 든 거래', rule_ids & gen_ids), ('룰 Alert 에만', rule_ids - gen_ids),
                                             ('생성기 Alert 에만', gen_ids - rule_ids)]]).set_index('거래')
tx_overlap

## 8. 저장

In [ ]:
result.to_csv(OUT / 'summary.csv')
ceiling.to_csv(OUT / 'ceiling.csv')
by_pattern.to_csv(OUT / 'by_pattern.csv'); by_size.to_csv(OUT / 'by_size.csv')
by_tail.to_csv(OUT / 'by_tail.csv'); by_pattern_tail.to_csv(OUT / 'by_pattern_tail.csv')
alerts.to_csv(OUT / 'alerts.csv'); combo.to_csv(OUT / 'rule_combos.csv'); largest.to_csv(OUT / 'largest.csv', index=False)
overlap.to_csv(OUT / 'overlap_alerts.csv'); tx_overlap.to_csv(OUT / 'overlap_tx.csv')
pd.concat([per.assign(method=name) for name, per in per_attempt.items()]).to_csv(OUT / 'per_attempt.csv', index=False)
groups[['edge_id', '룰', '대상', '묶음', 'r1', 'r2', 'y', 'pattern_type']].to_parquet(OUT / 'members.parquet', index=False)      # 룰 Alert 구성 거래 (R1 · R2)
wide[['edge_id', '룰', '대상', '묶음', '근거', 'r1', 'r2', 'y', 'pattern_type']].to_parquet(OUT / 'members_wide.parquet', index=False)   # 주변 거래까지 (R3 · R4)
extra.to_csv(OUT / 'extra.csv')
json.dump(dict(dataset=DATASET, rule_run=RULE_RUN, gen_run=GEN_RUN, test_window=[a, b], exclude=EXCLUDE,
               chosen=rule_run['chosen'], lonely=lonely),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))